# A4 데이터 분할 확인

`load_split()`이 돌려주는 `RatingSplit`의 다섯 표를 실제 데이터로 출력해 확인합니다.

| 표 | 내용 |
| --- | --- |
| `train` | 일반 독자 각자의 평점 앞 70% (학습) |
| `val` | 다음 15% (설정 고르기) |
| `test` | 마지막 15% (최종 채점) |
| `cold_tune` | 신규 독자 2,500명의 평점 전부 (신규 독자 튜닝) |
| `cold_final` | 신규 독자 2,500명의 평점 전부 (신규 독자 최종 채점) |

In [1]:
import pandas as pd
from bookrec.data import load_ratings, load_split

ratings = load_ratings()  # 원본 평점 (파일 순서 = 시간순)
split = load_split()      # 신규 독자 5,000명을 먼저 빼고, 나머지를 독자별 70/15/15로 나눈다
PARTS = ["train", "val", "test", "cold_tune", "cold_final"]
split

RatingSplit(train=         user_id  book_id  rating
0              1      258       5
1              2     4081       4
2              2      260       5
3              2     9296       5
4              2     2318       3
...          ...      ...     ...
5976418    49802     5218       3
5976419    49802     5659       3
5976420    49802     5760       3
5976421    49802     7112       3
5976422    49802     7583       3

[3769078 rows x 3 columns], val=         user_id  book_id  rating
918           73      618       4
919           73     5508       5
920           73     4707       4
921           73        2       4
958           73     3457       5
...          ...      ...     ...
5976433    49802     2303       4
5976434    49802     2341       3
5976435    49802     2550       3
5976436    49802     3166       4
5976437    49802     2549       3

[811075 rows x 3 columns], test=         user_id  book_id  rating
1169          73      118       4
1170          73       43       

## 1. 표마다 크기

In [2]:
# 표마다 평점 수, 독자 수, 독자 한 명의 평점 수(최소·중앙값·최대)
pd.DataFrame({
    name: {
        "평점 수": len(df),
        "독자 수": df["user_id"].nunique(),
        "독자당 최소": df.groupby("user_id").size().min(),
        "독자당 중앙값": df.groupby("user_id").size().median(),
        "독자당 최대": df.groupby("user_id").size().max(),
    }
    for name, df in ((name, getattr(split, name)) for name in PARTS)
}).T

,평점 수,독자 수,독자당 최소,독자당 중앙값,독자당 최대
train,3769078.0,48424.0,13.0,77.0,140.0
val,811075.0,48424.0,3.0,17.0,30.0
test,835308.0,48424.0,3.0,17.0,30.0
cold_tune,279907.0,2500.0,26.0,110.0,200.0
cold_final,281111.0,2500.0,24.0,111.0,196.0


## 2. 표마다 앞부분

In [3]:
for name in PARTS:
    print(f"--- {name}")
    display(getattr(split, name).head())  # 왼쪽 숫자는 원본에서의 줄 번호

--- train


,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


--- val


,user_id,book_id,rating
918,73,618,4
919,73,5508,5
920,73,4707,4
921,73,2,4
958,73,3457,5


--- test


,user_id,book_id,rating
1169,73,118,4
1170,73,43,3
1701,113,39,5
1702,113,110,4
1703,113,135,3


--- cold_tune


,user_id,book_id,rating
759,66,10,3
760,66,76,2
761,66,2,2
762,66,35,4
763,66,629,2


--- cold_final


,user_id,book_id,rating
707,62,572,5
708,62,171,3
709,62,1041,4
710,62,701,4
1590,112,4047,5


## 3. 독자 한 명이 어떻게 나뉘었나

일반 독자 한 명의 평점을 시간순으로 놓고, 각 평점이 어느 표에 들어갔는지 붙여 봅니다. 앞 70%는 train, 다음 15%는 val, 마지막 15%는 test여야 합니다.

In [4]:
user = split.train["user_id"].iloc[0]  # train의 첫 독자
part_of = pd.concat([getattr(split, name).assign(part=name) for name in ["train", "val", "test"]])["part"]
history = ratings[ratings["user_id"] == user].assign(part=part_of)  # 원본 순서 그대로 표 이름을 붙인다
print(f"독자 {user}: 평점 {len(history)}개 →", history["part"].value_counts().reindex(["train", "val", "test"]).to_dict())
history

독자 1: 평점 117개 → {'train': 81, 'val': 18, 'test': 18}


,user_id,book_id,rating,part
0,1,258,5,train
75,1,268,3,train
76,1,5556,3,train
77,1,3638,3,train
78,1,1796,5,train
...,...,...,...,...
5704475,1,142,4,test
5704476,1,642,4,test
5704477,1,901,4,test
5704479,1,212,3,test


## 4. 규칙 확인

In [5]:
cold = set(split.cold_tune["user_id"]) | set(split.cold_final["user_id"])
warm = set(split.train["user_id"])
whole = pd.concat([getattr(split, name) for name in PARTS]).sort_index()  # 다섯 표를 원래 줄 번호 순서로 합친다
pd.DataFrame([
    ("신규 독자 수 = 5,000 (튜닝 2,500 / 최종 2,500)", split.cold_tune["user_id"].nunique() == 2500 and split.cold_final["user_id"].nunique() == 2500),
    ("신규 독자가 train/val/test에 없음", not cold & (warm | set(split.val["user_id"]) | set(split.test["user_id"]))),
    ("튜닝용과 최종용 신규 독자가 겹치지 않음", not set(split.cold_tune["user_id"]) & set(split.cold_final["user_id"])),
    ("일반 독자 모두 train/val/test에 평점이 있음", warm == set(split.val["user_id"]) == set(split.test["user_id"])),
    ("다섯 표를 합치면 원본과 똑같음 (빠짐·중복 없음)", whole.equals(ratings)),
], columns=["확인", "결과"])

,확인,결과
0,"신규 독자 수 = 5,000 (튜닝 2,500 / 최종 2,500)",True
1,신규 독자가 train/val/test에 없음,True
2,튜닝용과 최종용 신규 독자가 겹치지 않음,True
3,일반 독자 모두 train/val/test에 평점이 있음,True
4,다섯 표를 합치면 원본과 똑같음 (빠짐·중복 없음),True
